# Landauer Energy Workflow

Parameterizable notebook for uncertainty-aware energy/work estimation from raw traces.

In [1]:
# Parameters
trace_path = "traces/run1.csv"
block_size = 5
cycle_column = "cycle"
typeb_json = "inputs/typeB.json"
sample_rate_hz = 10000
bootstrap_reps = 2000
alpha = 0.05
seed = 0
outputs_dir = "outputs"


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t as t_dist

required = {"t_s", "V", "I"}
df = pd.read_csv(trace_path)
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if cycle_column not in df.columns:
    raise ValueError(
        f"Missing cycle column '{cycle_column}'. Add a cycle ID per sample for robust integration."
    )

df = df.sort_values([cycle_column, "t_s"]).reset_index(drop=True)
df["P_W"] = df["V"] * df["I"]

E = (
    df.groupby(cycle_column, sort=True)
      .apply(lambda g: np.trapz(g["P_W"].to_numpy(), g["t_s"].to_numpy()))
      .rename("E_J")
      .reset_index()
)

if len(E) < 2:
    raise ValueError("Need at least two cycles to estimate Type-A uncertainty")

Evals = E["E_J"].to_numpy(dtype=float)
n = len(Evals)
rng = np.random.default_rng(seed)

def mbb_means(x: np.ndarray, reps: int, b: int) -> np.ndarray:
    n_ = len(x)
    b_ = max(1, min(int(b), n_))
    k = int(np.ceil(n_ / b_))
    idx = np.arange(n_)
    out = np.empty(reps, dtype=float)
    for r in range(reps):
      starts = rng.integers(0, n_ - b_ + 1, size=k)
      take = np.concatenate([idx[s:s+b_] for s in starts])[:n_]
      out[r] = x[take].mean()
    return out

boot_means = mbb_means(Evals, reps=int(bootstrap_reps), b=int(block_size))
mu = float(Evals.mean())
sigma_A = float(boot_means.std(ddof=1))

tb = {}
if typeb_json:
    typeb_path = Path(typeb_json)
    if typeb_path.exists():
        tb = json.loads(typeb_path.read_text(encoding="utf-8"))

rel2 = 0.0
for key in ["volt_gain_rel", "curr_gain_rel", "timing_rel"]:
    rel2 += float(tb.get(key, 0.0)) ** 2
sigma_B = abs(mu) * np.sqrt(rel2)

sigma_total = float(np.sqrt(sigma_A**2 + sigma_B**2))
dof = max(n - 1, 1)
tq = float(t_dist.ppf(1 - alpha / 2, dof))

ci_A = (mu - tq * sigma_A, mu + tq * sigma_A)
ci_B = (mu - tq * sigma_B, mu + tq * sigma_B)
ci_total = (mu - tq * sigma_total, mu + tq * sigma_total)

out_dir = Path(outputs_dir)
out_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([
    {
        "stat": "mean_energy_J",
        "value": mu,
        "sigma_A": sigma_A,
        "sigma_B": sigma_B,
        "sigma_total": sigma_total,
        "ci95_A_low": ci_A[0],
        "ci95_A_high": ci_A[1],
        "ci95_B_low": ci_B[0],
        "ci95_B_high": ci_B[1],
        "ci95_total_low": ci_total[0],
        "ci95_total_high": ci_total[1],
        "cycles": n,
        "block_size": int(max(1, min(int(block_size), n))),
        "bootstrap_reps": int(bootstrap_reps)
    }
])

summary.to_csv(out_dir / "energy-per-erasure.csv", index=False)
E.to_csv(out_dir / "cycle-energy.csv", index=False)

components = pd.DataFrame([
    {"component": "volt_gain_rel", "value": float(tb.get("volt_gain_rel", 0.0))},
    {"component": "curr_gain_rel", "value": float(tb.get("curr_gain_rel", 0.0))},
    {"component": "timing_rel", "value": float(tb.get("timing_rel", 0.0))}
])
components.to_csv(out_dir / "uncertainty-components.csv", index=False)

print(summary.to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: 'traces/run1.csv'